# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



In [29]:
!pip install scipy


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [30]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


## A.2. Missing values & Duplicate data

In [32]:
print(df.isnull().sum())
print(df.duplicated().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
1


## A.3. Invalid values

In [33]:
print(df.isna().sum())

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [34]:
def phanloai(bmi):
    if bmi <25:
        return 'Normal'
    elif bmi >= 25 or bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_group'] = df['bmi']
df[['bmi', 'bmi_group']].head()


,bmi,bmi_group
0,27.900,27.900
1,33.770,33.770
2,33.000,33.000
3,22.705,22.705
4,28.880,28.880


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [35]:
print(df.mean(numeric_only=True))
print(df.median(numeric_only=True))

age             39.207025
bmi             30.663397
children         1.094918
charges      13270.422265
bmi_group       30.663397
dtype: float64
age            39.000
bmi            30.400
children        1.000
charges      9382.033
bmi_group      30.400
dtype: float64


## Group 2 — Dispersion

In [36]:
num_df = df.select_dtypes(include=['int', 'float'])
range = num_df.max() - num_df.min()
var = num_df.var()
std = num_df.std()
IQR = num_df.quantile(0.75) - num_df.quantile(0.25)

## Group 3 — Location and Shape

In [37]:
Q1 = num_df.quantile(0.25)
Q2 = num_df.quantile(0.50)
Q3 = num_df.quantile(0.75)
skew = num_df.skew()
kurt = num_df.kurt()



---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [ ]:
co_hut = df[df['smoker'] == 'yes']
khong_hut = df[df['smoker'] == 'no']
cp_co_hut = co_hut['charges']
cp_khong_hut = khong_hut['charges']
print('Chi phí của người có hút thuốc: ', cp_co_hut.mean())
print('Chi phí của người không hút thuốc: ', cp_khong_hut.mean())
print('Mức chênh: ', cp_co_hut.mean() / cp_khong_hut.mean())
print('Các vùng có hút: ', co_hut.groupby('region')['charges'].mean())
print('Các vùng không hút: ', khong_hut.groupby('region')['charges'].mean())



Chi phí của người có hút thuốc:  63770.42801
Chi phí của người không hút thuốc:  36910.60803
Mức chênh:  1.7276992012206633
Các vùng có hút:  region
northeast    58571.07448
northwest    60021.39897
southeast    63770.42801
southwest    52590.82939
Name: charges, dtype: float64
Các vùng không hút:  region
northeast    32108.66282
northwest    33471.97189
southeast    36580.28216
southwest    36910.60803
Name: charges, dtype: float64


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [39]:

#tuong quan voi nguoi co hut thuoc
print('Có hút: ', df['bmi'].corr(co_hut['charges']))
print('Không hút: ', df['bmi'].corr(khong_hut['charges']))

Có hút:  0.8064806070155401
Không hút:  0.0840365431283327


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [40]:
cp_theo_vung = df.groupby('region')['charges'].mean()
print('Vùng có chi phí cao nhất là', cp_theo_vung.idxmax(), 'với số tiền: ',cp_theo_vung.max())




Vùng có chi phí cao nhất là southeast với số tiền:  14735.41143760989


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [41]:
cp_tb = df['charges'].mean()
con_gai =df[df['sex'] !='female']
cp_tb_khi_khong_co_con_gai =con_gai['charges'].mean()
print('CP TB ALL: ', cp_tb)
print('CP TB Khi không con gái', cp_tb_khi_khong_co_con_gai)
print('Con gái không làm tăng chi phí')

CP TB ALL:  13270.422265141257
CP TB Khi không con gái 13956.751177721893
Con gái không làm tăng chi phí


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [42]:
print(df['age'].corr(df['charges']))
print('Tuổi không có tương quan với chi phí bảo hiểm')


0.29900819333064776
Tuổi không có tương quan với chi phí bảo hiểm


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*